# Adaptive Load Balancing Analysis

This notebook contains detailed analysis of the reinforcement learning load balancer performance.

In [ ]:
import sys
import os
sys.path.append('src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from stable_baselines3 import PPO

from environment import LoadBalancerEnv
from agent import evaluate_rl_agent
from baselines import run_baseline_comparison

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Environment Setup and Validation

In [ ]:
# Create environment
env = LoadBalancerEnv(n_servers=3, max_steps=100)
print("Environment created successfully!")
print(f"State space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.n}")

# Test random actions
obs, _ = env.reset()
total_reward = 0
for _ in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"Random policy total reward: {total_reward:.2f}")

## 2. Baseline Algorithm Comparison

In [ ]:
# Run baseline comparison
baseline_results = run_baseline_comparison(env, n_episodes=5)

# Display results
df_baselines = pd.DataFrame.from_dict(baseline_results, orient='index')
df_baselines.round(3)

In [ ]:
# Plot baseline comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Latency comparison
algorithms = list(baseline_results.keys())
latencies = [baseline_results[alg]['avg_latency'] for alg in algorithms]
ax1.bar(algorithms, latencies, color=['#e74c3c', '#e67e22', '#f39c12', '#27ae60'])
ax1.set_title('Average Latency by Algorithm')
ax1.set_ylabel('Latency (ms)')
ax1.tick_params(axis='x', rotation=45)

# Utilization comparison
utilizations = [baseline_results[alg]['avg_utilization'] for alg in algorithms]
ax2.bar(algorithms, utilizations, color=['#e74c3c', '#e67e22', '#f39c12', '#27ae60'])
ax2.set_title('Average Server Utilization by Algorithm')
ax2.set_ylabel('CPU Utilization')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('results/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. RL Agent Evaluation

In [ ]:
# Load trained model
try:
    model = PPO.load("models/rl_load_balancer")
    print("Model loaded successfully!")
    
    # Evaluate RL agent
    rl_results = evaluate_rl_agent(env, model, n_episodes=5)
    print("RL Agent Performance:")
    for metric, value in rl_results.items():
        print(f"  {metric}: {value:.3f}")
        
except FileNotFoundError:
    print("No trained model found. Please run training first.")
    rl_results = None

## 4. Statistical Analysis

In [ ]:
if rl_results:
    # Combine all results
    all_results = baseline_results.copy()
    all_results['RL Agent'] = rl_results
    
    # Create comparison table
    df_all = pd.DataFrame.from_dict(all_results, orient='index')
    df_all = df_all[['avg_latency', 'avg_utilization', 'std_latency']]
    df_all.columns = ['Avg Latency (ms)', 'Avg Utilization', 'Latency Std Dev']
    
    print("Complete Performance Comparison:")
    print(df_all.round(3))
    
    # Calculate improvements
    rr_latency = all_results['Round Robin']['avg_latency']
    rl_latency = all_results['RL Agent']['avg_latency']
    improvement = ((rr_latency - rl_latency) / rr_latency) * 100
    
    print(f"\nRL Improvement over Round Robin: {improvement:.1f}%")

## 5. Traffic Pattern Analysis

In [ ]:
# Simulate different traffic patterns
def simulate_traffic_pattern(env, model, pattern_name, pattern_func):
    """Simulate a specific traffic pattern."""
    obs, _ = env.reset()
    latencies = []
    
    for step in range(200):
        # Modify request load based on pattern
        base_load = pattern_func(step)
        
        # Get action from model
        action, _ = model.predict(obs, deterministic=True)
        
        # Step environment
        obs, reward, terminated, truncated, _ = env.step(action)
        latency = -reward  # Convert reward to latency
        latencies.append(latency)
        
        if terminated or truncated:
            break
    
    return latencies

# Define traffic patterns
def uniform_traffic(step):
    return np.random.uniform(0.05, 0.3)

def bursty_traffic(step):
    if 50 <= step <= 100:
        return np.random.uniform(0.1, 0.6)  # Burst period
    else:
        return np.random.uniform(0.05, 0.2)  # Normal period

if rl_results:
    # Run simulations
    uniform_latencies = simulate_traffic_pattern(env, model, "Uniform", uniform_traffic)
    bursty_latencies = simulate_traffic_pattern(env, model, "Bursty", bursty_traffic)
    
    # Plot comparison
    plt.figure(figsize=(12, 6))
    plt.plot(uniform_latencies, label='Uniform Traffic', alpha=0.7)
    plt.plot(bursty_latencies, label='Bursty Traffic', alpha=0.7)
    plt.axvspan(50, 100, alpha=0.1, color='red', label='Burst Period')
    plt.title('RL Agent Performance Under Different Traffic Patterns')
    plt.xlabel('Time Step')
    plt.ylabel('Response Time (ms)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('results/traffic_patterns.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Server State Analysis

In [ ]:
def analyze_server_states(env, model, n_steps=100):
    """Analyze how server states evolve over time."""
    obs, _ = env.reset()
    server_histories = [[] for _ in range(env.n_servers)]
    
    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        
        # Record server states
        for i in range(env.n_servers):
            cpu = obs[i*3]
            queue = obs[i*3 + 1] * env.max_queue  # Denormalize
            latency = obs[i*3 + 2] * 200  # Denormalize
            server_histories[i].append((cpu, queue, latency))
            
        if terminated or truncated:
            break
    
    return server_histories

if rl_results:
    # Analyze server states
    histories = analyze_server_states(env, model)
    
    # Plot server CPU utilization over time
    plt.figure(figsize=(12, 8))
    for i, history in enumerate(histories):
        cpus = [h[0] for h in history]
        plt.plot(cpus, label=f'Server {i+1}', linewidth=2)
    
    plt.title('Server CPU Utilization Over Time (RL Agent)')
    plt.xlabel('Time Step')
    plt.ylabel('CPU Utilization')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('results/server_utilization.png', dpi=150, bbox_inches='tight')
    plt.show()